# Data Ingestion with Lakeflow Connect

Training notebook covering Lakeflow Connect: fully-managed connectors that ingest data from enterprise applications, databases, and cloud storage into Unity Catalog — with incremental ingestion, schema evolution, and orchestration built in.

## 📋 What We'll Cover

1. **What is Lakeflow Connect** — purpose, capabilities, and where it fits in the Databricks platform
2. **Connector Architecture** — SaaS, database, and file connector components
3. **Supported Connectors** — enterprise applications, databases, cloud storage, message buses
4. **Incremental Ingestion & CDC** — how Lakeflow Connect tracks and replays changes efficiently
5. **Setting Up a Connector** — UI walkthrough: connection, pipeline, destination tables
6. **Orchestration** — schedules, jobs, and chaining tasks
7. **Monitoring** — pipeline status, metrics, event logs

## 1. What is Lakeflow Connect

Lakeflow Connect is Databricks' **managed ingestion service** that moves data from external systems into Unity Catalog with minimal setup. It handles connector management, scheduling, schema evolution, and incremental change tracking so you don't need to maintain bespoke ETL code.

### Where it fits

```
 External Systems                Databricks Platform
 ┌──────────────────┐            ┌─────────────────────────────┐
 │  Salesforce      │            │                             │
 │  SAP             │──Lakeflow──▶  Unity Catalog Tables       │
 │  Workday         │  Connect   │  (Bronze / Silver / Gold)   │
 │  PostgreSQL      │            │                             │
 │  Kafka           │            └─────────────────────────────┘
 │  Azure Blob / S3 │
 └──────────────────┘
```

### Key capabilities

| Capability | Description |
|---|---|
| **Fully managed** | No infrastructure to provision or maintain |
| **Incremental ingestion** | Only changed records are loaded after the initial full sync |
| **CDC support** | Captures inserts, updates, and deletes from databases |
| **Schema evolution** | Automatically adapts when source schemas change |
| **Unity Catalog native** | Tables land directly in UC with lineage, access control, and governance |
| **Orchestration built-in** | Schedules automatically create Lakeflow Jobs |

### Supported interfaces

You can configure and manage Lakeflow Connect via the **Databricks UI**, Declarative Automation Bundles (DABs), Databricks APIs, SDKs, and CLI.

## 2. Connector Architecture

Lakeflow Connect has three connector families, each with slightly different components.

### SaaS & File Connectors (e.g. Salesforce, Workday, Azure Blob)

```
 ┌──────────────┐    ┌────────────────┐    ┌──────────────────────┐
 │  Connection  │───▶│ Ingestion      │───▶│ Destination Tables   │
 │ (credentials)│    │ Pipeline       │    │ in Unity Catalog     │
 └──────────────┘    └────────────────┘    └──────────────────────┘
```

* **Connection** — stores credentials and endpoint details securely (backed by Databricks secrets)
* **Ingestion Pipeline** — the Lakeflow pipeline that reads from the source and writes to UC
* **Destination Tables** — Delta tables created/updated in the chosen catalog and schema

### Database Connectors (e.g. PostgreSQL, SQL Server, MySQL)

```
 ┌──────────────┐    ┌─────────────────┐    ┌──────────────┐    ┌───────────────┐
 │  Connection  │───▶│ Ingestion       │───▶│  Staging     │───▶│  Destination  │
 │              │    │ Gateway (job)   │    │  Storage     │    │  Tables in UC │
 └──────────────┘    └─────────────────┘    └──────────────┘    └───────────────┘
```

Database connectors add an **Ingestion Gateway** (a continuous Lakeflow Job) and **staging storage** (cloud object store) to support continuous CDC.

### File / Cloud Storage Connectors

Read files directly from Azure Blob, ADLS, S3, or GCS into Delta tables. Uses Auto Loader under the hood with managed checkpointing.

## 3. Supported Connectors

### Enterprise Applications (SaaS)

| Connector | Incremental Support | Notes |
|---|---|---|
| Salesforce | Yes | Cursor-based; uses `SystemModstamp` |
| Workday | Yes | Report-based via Workday RaaS API |
| SAP | Yes | SAP OData or ABAP CDS views |
| ServiceNow | Yes | `sys_updated_on` cursor |
| Google Analytics | Yes | Data API |
| HubSpot | Yes | CRM API |

### Databases

| Connector | CDC | Notes |
|---|---|---|
| PostgreSQL | Yes | Logical replication (pgoutput) |
| MySQL | Yes | Binlog CDC |
| SQL Server | Yes | Change tracking + CDC |
| Oracle | Yes | LogMiner CDC |
| MongoDB | Yes | Change streams |

### Cloud Storage & Files

| Source | Format | Notes |
|---|---|---|
| Azure Blob / ADLS | CSV, JSON, Parquet, Avro | Auto Loader backed |
| Amazon S3 | CSV, JSON, Parquet, Avro | Auto Loader backed |
| Google Cloud Storage | CSV, JSON, Parquet | Auto Loader backed |

### Message Buses

| Connector | Notes |
|---|---|
| Apache Kafka | Structured Streaming under the hood |
| Azure Event Hubs | Kafka-compatible endpoint |
| Confluent Cloud | Managed Kafka |

> 💡 **Tip:** In the Databricks UI, navigate to **Data Engineering → Lakeflow Connect** to see the connectors available in your workspace.

## 4. Incremental Ingestion & CDC

### How Incremental Ingestion Works

```
Run 1 (full load)
 Source: 1,000,000 rows ──────────────────────────▶ UC Table: 1,000,000 rows
                          ↓ records checkpoint

Run 2 (incremental)
 Source: only 5,000 changed rows ─────────────────▶ UC Table: upserted / appended
```

**First run:** performs a full load of all selected tables/objects.
**Subsequent runs:** only records that changed since the last run are ingested.

| Source Type | Change Tracking Mechanism |
|---|---|
| SaaS (Salesforce, Workday) | Cursor column (e.g. `LastModifiedDate`) |
| Relational DB (PostgreSQL, MySQL) | CDC via transaction log / binlog |
| Cloud storage | Auto Loader file-level tracking |

### Change Data Capture (CDC) — MERGE semantics

For database connectors, CDC captures every DML operation:

| CDC event | Action in UC table |
|---|---|
| `INSERT` | New row added |
| `UPDATE` | Existing row updated (upserted) |
| `DELETE` | Row removed (or soft-deleted with a flag) |

The UC table always reflects the current state of the source — no duplicate rows from re-running.

### Schema Evolution

When a new column is added at the source, Lakeflow Connect automatically adds that column to the destination Delta table on the next run. Renamed or dropped columns are handled based on connector configuration.

## 5. Setting Up a Connector (UI Walkthrough)

The setup wizard in the Databricks UI walks you through four steps.

### Step 1 — Create a Connection
1. Go to **Catalog → External Locations & Connections → Create Connection**.
2. Choose the connector type (e.g. Salesforce).
3. Enter the authentication details (OAuth, username/password, API key, etc.).
4. Databricks stores credentials in a managed secret scope.

### Step 2 — Create an Ingestion Pipeline
1. Navigate to **Data Engineering → Lakeflow Connect → Create pipeline**.
2. Select your connection.
3. Choose which objects/tables/topics to ingest.
4. Set the **destination catalog and schema** in Unity Catalog.

### Step 3 — Configure Destination Tables
* Each source object maps to one Delta table.
* Choose write mode: **Append** (raw events) or **Apply changes** (CDC upsert).
* Optionally rename columns or set primary keys.

### Step 4 — Schedule & Run
1. Set a schedule (e.g. every 15 minutes, hourly, daily).
2. Lakeflow Connect creates a **Lakeflow Job** for the schedule automatically.
3. Click **Start** for the first full load.

> ⚠️ **Note:** For database connectors, an **Ingestion Gateway** must also be configured and deployed to a compute resource with network access to the source database.

## 6. Orchestration

Each schedule you add to a Lakeflow Connect pipeline automatically creates a **Lakeflow Job**. You can chain additional tasks to that job.

### Trigger options

| Trigger | Description |
|---|---|
| **Scheduled (cron)** | Run at fixed intervals, e.g. every hour |
| **Continuous** | Keep the pipeline running permanently (streaming CDC) |
| **Manual** | Run on demand via UI, API, or CLI |

### Chaining tasks

```
Job: daily_customer_sync
 ├── Task 1: Lakeflow Connect ingestion  (ingest customers from Salesforce)
 ├── Task 2: Notebook transformation     (depends on Task 1)
 └── Task 3: Dashboard refresh           (depends on Task 2)
```

You can add extra tasks to the auto-generated job in the **Lakeflow Jobs** UI or via Declarative Automation Bundles.

## 7. Monitoring

### Where to Look

| Location | What You See |
|---|---|
| **Lakeflow Connect UI** | Pipeline status, last run time, rows ingested, error details |
| **Lakeflow Jobs UI** | Job run history, task durations, failure messages |
| **Unity Catalog lineage** | Upstream source → destination tables, column-level lineage |
| **Event log** | Detailed per-run events (schema changes, row counts, errors) |

### Key Metrics to Watch

| Metric | Healthy Signal |
|---|---|
| **Rows ingested per run** | Consistent with expected change volume |
| **Run duration** | Stable; large increases suggest source growth or connectivity problems |
| **Lag (CDC mode)** | Low lag means the pipeline is keeping up |
| **Errors / warnings** | Zero schema drift warnings in steady state |

### Alerting

Set up Lakeflow Job notifications (email / webhook) on the auto-generated job to alert on failure or when a run exceeds a duration threshold.